# v3.3 iron-man run — the locked version against BC, self-hosted klein on an A100

One model. Every klein call is timed; `run/meta/cost.json` totals the run against the hourly rate you set below and beside the fal-equivalent price. **Nothing to upload** — cell 3 pulls the bundle from GitHub. Run cells in order. **Runtime → A100.**

Open this notebook directly: https://colab.research.google.com/github/101011101/magichour_takehome/blob/v3.3-lock/v3/colab/v33_ironman.ipynb

In [ ]:
# 1 · settings
A100_USD_PER_HOUR = 1.20      # what this runtime costs you per hour; edit. Colab Pro A100 ≈ 11.8 CU/h at $0.10/CU.
SEEDS = [46, 47, 48]          # the version is scored at more than one seed; 46 is the seed every V3 run used
ARMS = ("BC", "V")
LIMIT = None                  # e.g. 20 for a partial run; None = the whole matrix
DRIVE_PROJECT_DIR = "Side projects and shi"   # exact Drive folder name; the UI truncates it

In [ ]:
# 2 · Drive: find the HF cache that actually holds klein, and a place to keep the zip
import os
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'
BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
KLEIN = 'models--black-forest-labs--FLUX.2-klein-4B'
candidates = [os.path.join(BASE, 'tryon_models', 'hf_cache'),   # where V2's notebook put it
              os.path.join(MYDRIVE, 'hf_cache'),                 # a root-level hf_cache, if that is where it lives
              os.path.join(BASE, 'hf_cache')]
found = [c for c in candidates if os.path.isdir(os.path.join(c, 'hub', KLEIN))]
HF_HOME = found[0] if found else candidates[0]
os.environ['HF_HOME'] = HF_HOME                                   # must be set before any diffusers import
os.environ['V3_MODEL_DIR'] = os.path.join(BASE if os.path.isdir(BASE) else MYDRIVE, 'v3_models')
os.makedirs(HF_HOME, exist_ok=True); os.makedirs(os.environ['V3_MODEL_DIR'], exist_ok=True)
for c in candidates:
    print(('  klein here  ' if c in found else '  -           ') + c)
print('HF_HOME =', HF_HOME, '(klein cached)' if found else '(no cached klein found - cell 4 downloads ~13 GB here once)')
if len(found) > 1: print('NOTE: klein is cached in more than one place; the first is used. The others are duplicates you can delete.')

In [ ]:
# 3 · install, and pull the bundle straight from GitHub (public repo, branch v3.3-lock)
!pip -q install -U diffusers transformers accelerate sentencepiece protobuf mediapipe onnxruntime-gpu opencv-python-headless
BRANCH = 'v3.3-lock'
!cd /content && rm -rf ironman && wget -q -O v33_ironman_bundle.zip https://github.com/101011101/magichour_takehome/raw/{BRANCH}/v33_ironman_bundle.zip && unzip -qo v33_ironman_bundle.zip -d ironman
%cd /content/ironman
import os, onnxruntime as ort, torch
assert os.path.exists('lib/run_ironman.py'), 'bundle did not unpack - check the branch name and that the repo is public'
print('onnxruntime providers:', ort.get_available_providers(), '- BiRefNet needs CUDAExecutionProvider here; on CPU each crop is ~50 s')
print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
# 4 · weights: load once (from the Drive cache; downloads there if absent) and time it
import sys; sys.path.insert(0, 'lib')
import klein_local as K
K.load(); K.info()

In [ ]:
# 5 · one pair first — check run/gen has both arms before spending on the matrix
import run_ironman as R
R.main('matrix.csv', 'testset', limit=1, seeds=SEEDS[:1], arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
import os; print(sorted(os.listdir('run/gen')))

In [ ]:
# 6 · the whole matrix, every seed. Resumable: re-run this cell after any interruption.
R.main('matrix.csv', 'testset', limit=LIMIT, seeds=SEEDS, arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
import json; print(json.dumps(json.load(open('run/meta/cost.json')), indent=1))

In [ ]:
# 7 · zip the evidence and copy it to Drive; download from there or from the Files pane
import shutil, time
name = f"v33_ironman_run_{time.strftime('%Y%m%d_%H%M')}"
shutil.make_archive(f'/content/{name}', 'zip', 'run')
dst = os.path.join(BASE, 'v3_runs', name + '.zip'); os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy(f'/content/{name}.zip', dst)
print('zip:', dst)
print('then locally:  python3 v3/build/ironman_page.py', name + '.zip')

## BC repair (2026-08-30)

The first run's `BC` kept the bald **head** (bald → A4 crop). V2's `BC_klein` subtracts the head with the V2 cropper, which lives in `v2/build` and is not in this bundle. Repair in three steps: **(8)** regenerate the 56 bald frames here and send them to Drive; **locally** `python3 v3/build/ironman_bc_crop.py <dir>` makes `refs/{g}__BC.jpg` with the V2 cropper; **(9)** bring those back and run the 600 BC edits. The `V` outputs are untouched.

In [ ]:
# 8 · BC repair, step 1: reuse the previous run (from Drive) and make the 56 bald frames on the A100
PREV_ZIP = os.path.join(BASE, 'v3_runs', 'v33_ironman_run_20260830_0548.zip')   # the run being repaired
import zipfile, shutil
if not os.path.isdir('run/gen'):
    zipfile.ZipFile(PREV_ZIP).extractall('run'); print('previous run unpacked')
for f in os.listdir('run/refs'):                        # retire the head-kept BC references
    if f.endswith('__BC.jpg'): os.rename('run/refs/'+f, 'run/refs/'+f.replace('__BC.jpg','__BCA4.jpg'))
for f in os.listdir('run/gen'):
    if '__BC__' in f: os.rename('run/gen/'+f, 'run/gen/'+f.replace('__BC__','__BCA4__'))
import run_ironman as R
R.main('matrix.csv', 'testset', limit=LIMIT, seeds=SEEDS, arms=('BC',), gpu_usd_per_hour=A100_USD_PER_HOUR, stage='bald')
name = 'v33_ironman_bald_20260830_0548'
shutil.make_archive('/content/'+name, 'zip', 'run/refs', None)   # small: bald frames + previous refs
shutil.copy('/content/'+name+'.zip', os.path.join(BASE, 'v3_runs', name+'.zip'))
print('bald frames ->', os.path.join(BASE, 'v3_runs', name+'.zip'), ' now run the V2 cropper locally')

In [ ]:
# 9 · BC repair, step 2: after the local cropper, upload v33_ironman_bcrefs.zip to /content, then the 600 BC edits
BCREFS_ZIP = '/content/v33_ironman_bcrefs.zip'
assert os.path.exists(BCREFS_ZIP), 'upload v33_ironman_bcrefs.zip (refs/{g}__BC.jpg from ironman_bc_crop.py) to /content'
zipfile.ZipFile(BCREFS_ZIP).extractall('run/refs')
print(sum(f.endswith('__BC.jpg') for f in os.listdir('run/refs')), 'BC references in place')
R.main('matrix.csv', 'testset', limit=LIMIT, seeds=SEEDS, arms=('BC',), gpu_usd_per_hour=A100_USD_PER_HOUR, stage='bcedit')
name = f"v33_ironman_run_20260830_0548_bcfix_{time.strftime('%H%M')}"
shutil.make_archive('/content/'+name, 'zip', 'run'); shutil.copy('/content/'+name+'.zip', os.path.join(BASE, 'v3_runs', name+'.zip'))
print('repaired run ->', os.path.join(BASE, 'v3_runs', name+'.zip'))

## v3.4 link A — no ankle cut, on the v3.3 failure set

`v34_failures.csv` is the 31 pairs where v3.3 had a failing cell on the iron-man run. Arm `Vnc` is the locked version with the ankle cut removed, everything else identical; edited at the same three seeds. Reuses the previous run's inputs and A4 crops.

In [ ]:
# 10 · v3.4 link A: Vnc on the failure set (references regenerated without the cut; ~31 pairs x 3 seeds)
import zipfile, shutil, os, time
PREV_ZIP = os.path.join(BASE, 'v3_runs', 'v33_ironman_run_20260830_0548.zip')
if not os.path.isdir('run/gen'):
    zipfile.ZipFile(PREV_ZIP).extractall('run'); print('previous run unpacked')
import run_ironman as R
R.main('v34_failures.csv', 'testset', limit=None, seeds=SEEDS, arms=('Vnc',), gpu_usd_per_hour=A100_USD_PER_HOUR)
name = f"v34_linkA_nocut_{time.strftime('%Y%m%d_%H%M')}"
os.makedirs('/content/out', exist_ok=True)
with zipfile.ZipFile(f'/content/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('run/refs'):
        if '__Vnc' in f: z.write('run/refs/'+f, 'refs/'+f)
    for f in os.listdir('run/gen'):
        if '__Vnc__' in f: z.write('run/gen/'+f, 'gen/'+f)
    for f in ('prompts.json', 'timings.csv', 'cost.json', 'run.json'): z.write('run/meta/'+f, 'meta/'+f)
shutil.copy(f'/content/{name}.zip', os.path.join(BASE, 'v3_runs', name+'.zip')); print('->', os.path.join(BASE, 'v3_runs', name+'.zip'))